# 01 — Explore ECOSoundSet
Inspect the dataset, understand class distribution, and identify which UK Orthoptera species have sufficient clips.

**Kernel:** `Python (orthoptera-training)`  
**Dataset:** `datasets/ecosoundset/` — download via `zenodo_get 15043892` if not present.

In [ ]:
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent  # cwd relative to current notebook path
ECO_ROOT = PROJECT_ROOT / "datasets" / "ecosoundset"
INSECT_ROOT = PROJECT_ROOT / "datasets" / "insectset459"


## EcoSoundSet Exploration

In [ ]:
# ── Load annotations and filter to train split ────────────────────────────────
all_annot_eco = pd.read_csv(ECO_ROOT / "annotated_audio_segments.csv")
train = all_annot_eco[all_annot_eco["subset"] == "train"].copy()
print(f"Total clips (train): {all_annot_eco['audio_segment_file_name'].nunique()}")
print(f"Total annotations (train): {len(train)}")
print(f"Columns: {list(train.columns)}")
print(f"Annotation Categories: {list(train['label_category'].unique())}")
train.head()


In [ ]:
# ── Quantify Overlapping Annotations ──────────────────────────────────────────
labels_per_file = all_annot_eco.groupby("audio_segment_file_name")["label"].count()
print(f"{'Clips with overlapping annotations:':<35} {labels_per_file[labels_per_file > 1].count()}")
print(f"{'Average annotations per file:':<35} {labels_per_file.mean()}")
print(f"{'Least annotations per file:':<35} {labels_per_file.min()}")
print(f"{'Least annotated file:':<35} {labels_per_file.idxmin()}")
print(f"{'Maximum annotations per file:':<35} {labels_per_file.max()}")
print(f"{'Most annotated file:':<35} {labels_per_file.idxmax()}")


In [ ]:
# ── All Orthoptera species in the train split ────────────────────────────────
orth_eco = train[train["label_category"] == "Orthoptera"]
print(f"Orthoptera species: {orth_eco['label'].nunique()}\n")
print(f"Annotations per species:\n\n{orth_eco['label'].value_counts().to_string()}")


In [ ]:
# ── Filter to UK target species ───────────────────────────────────────────────
# ECOSoundSet uses trinomial names; Meconema thalassinum is absent from both datasets.
UK_SPECIES = [
    "Chorthippus brunneus brunneus",           # Field Grasshopper
    "Pseudochorthippus parallelus parallelus", # Meadow Grasshopper
    "Omocestus viridulus",                     # Common Green Grasshopper
    "Tettigonia viridissima",                  # Great Green Bush-cricket
    "Roeseliana roeselii",                     # Roesel's Bush-cricket
    "Pholidoptera griseoaptera",               # Dark Bush-cricket
    "Leptophyes punctatissima",                # Speckled Bush-cricket
    "Gryllus campestris",                      # Field Cricket
]

orth_uk_eco = orth_eco[orth_eco["label"].isin(UK_SPECIES)].copy()
eco_file_index = {file_path.name: file_path for file_path in ECO_ROOT.rglob("*.wav")}
orth_uk_eco["filepath"] = orth_uk_eco["audio_segment_file_name"].map(eco_file_index)


In [ ]:
# ─── Derive unique sample rates ────────────────────
srs = {}
for fp in orth_uk_eco["filepath"].unique():
    y, sr = librosa.load(fp, sr=None)
    if sr in srs:
        srs[sr] += 1
    else:
        srs[sr] = 1

print("Sample Rates:\n")
for sr in srs:
    print(f"{sr}hz: {srs[sr]}")


In [ ]:
# ── Quantify total Orthoptera annotations and clips ───────────────────────────
# Multiple annotations can appear per clips. Annotation count does not equal clip count.
print(f"ECOSoundSet UK species annotations: {len(orth_uk_eco)}")
print(f"ECOSoundSet UK species clips: {orth_uk_eco['audio_segment_file_name'].nunique()}")
labels_per_file_orth = orth_uk_eco.groupby("audio_segment_file_name")["label"].count()
print(f"{'Average annotations per file:':<30} {labels_per_file_orth.mean()}")
print(f"\nAnnotations per species:\n\n{orth_uk_eco['label'].value_counts().to_string()}")
print(f"\nFiles per species:\n\n{orth_uk_eco.groupby('label')['audio_segment_file_name'].nunique()}")

# ── Quantify overlapping Orthoptera annotations ───────────────────────────────
# Files with at least one UK target species annotation.
uk_files = set(orth_uk_eco["audio_segment_file_name"])

# Files with at least 2 distinct Orthoptera annotations.
multi_orth = all_annot_eco[all_annot_eco["label_category"] == "Orthoptera"].groupby("audio_segment_file_name")["label"].nunique()
multi_orth_files = multi_orth[multi_orth > 1].index

# Files in both.
mixed_files = uk_files.intersection(multi_orth_files)

print(f"\n{'Clips with distinct overlapping Orthoptera annotations:'} {len(mixed_files)}")


In [ ]:
# ── Visualise class balance ───────────────────────────────────────────────────
counts = orth_uk_eco["label"].value_counts()
short_names = [s.split()[-1] for s in counts.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(short_names, counts.values, color="#2d8a4e")
ax.set_xlabel("Annotations")
ax.set_title("UK Orthoptera — Annotations per Species (ECOSoundSet train split)")
ax.axvline(100, color="orange", linestyle="--", linewidth=1, label="100 annotation threshold")
ax.legend()
plt.tight_layout()
plt.show()

low = counts[counts < 50]
if len(low):
    print(f"\n⚠ Species with <50 annotations (consider InsectSet459 supplement):")
    print(low.to_string())


In [ ]:
# ── Illuminate avg, min, and max range of frequencies across Orthoptera annotations ─────
freq_avg_eco = orth_uk_eco.groupby("label")[["annotation_min_freq", "annotation_max_freq"]].mean()
freq_avg_eco.rename(
    columns={"annotation_min_freq": "Avg Minimum Frequency", "annotation_max_freq": "Avg Maximum Frequency"}, 
    inplace=True
)
print(f"{freq_avg_eco.to_string()}\n")

freq_min_eco = orth_uk_eco.groupby("label")["annotation_min_freq"].agg(["min", "idxmin"])
freq_min_eco.rename(columns={"min": "Minimum Frequency", "idxmin": "Row Index"}, inplace=True)
# Minimum Frequencies of 0 are due to incomplete labeling where min and max frequencies represent the full sample rate range.
print(f"{freq_min_eco}\n")

freq_max_eco = orth_uk_eco.groupby("label")["annotation_max_freq"].agg(["max", "idxmax"])
freq_max_eco.rename(columns={"max": "Maximum Frequency", "idxmax": "Row Index"}, inplace=True)
print(freq_max_eco)


In [ ]:
# ── Inspect a clip — spectrogram preview ─────────────────────────────────────
# Set use_random to True for a randomised UK Orthoptera sample
# Set use_random to False to select any annotation from ECOSoundSet
use_random = False
selected_index = 77508  # CSV row number minus 1

if use_random:
    sample = orth_uk_eco.sample(1).iloc[0]
else:
    sample = all_annot_eco.loc[selected_index]

audio_path = next(ECO_ROOT.rglob(sample["audio_segment_file_name"]), None)

if audio_path and audio_path.exists():
    t_start = 0
    t_end = min(4, sample["audio_segment_final_time"] - sample["audio_segment_initial_time"])
    y, sample_rate = librosa.load(audio_path, sr=None, offset=t_start, duration=t_end - t_start)
    print(f"{'Species:':<20}{sample['label']}")
    print(f"{'File:':<20}{sample['audio_segment_file_name']}")
    print(f"{'Annotation Start:':<20}{sample['annotation_initial_time']}")
    print(f"{'Annotation End:':<20}{sample['annotation_final_time']}")
    print(f"{'Sample Rate:':<20}{sample_rate} Hz")
    print(f"{'Min Frequency:':<20}{sample['annotation_min_freq']} Hz")
    print(f"{'Max Frequency:':<20}{sample['annotation_max_freq']} Hz")
    print(f"{'Duration:':<20}{sample['annotation_final_time'] - sample['annotation_initial_time']:.1f} s")

    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    librosa.display.waveshow(y, sr=sample_rate, ax=axes[0])
    axes[0].set_title(f"{sample['label']} — waveform")

    n_fft = min(2048, len(y))
    S = librosa.feature.melspectrogram(y=y, sr=sample_rate, n_mels=128, fmax=sample_rate/2, n_fft=n_fft)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sample_rate, x_axis="time", y_axis="mel",
                             fmax=sample_rate/2, ax=axes[1], cmap="viridis")
    axes[1].set_title("Mel spectrogram")
    plt.colorbar(axes[1].collections[0], ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()
else:
    print(f"Audio file not found: {audio_path}")


## InsectSet459 Exploration

In [ ]:
# ── Load annotations and filter to train split ────────────────────────────────
all_annot_insect = pd.read_csv(INSECT_ROOT / "InsectSet459_Train_Val_Annotation.csv")
# Use train and validation subset as training data, use ECO val and test as definitive evaluation sets.
print(f"Total clips: {all_annot_insect['file_name'].nunique()}")
print(f"Total annotations (train and validation): {len(all_annot_insect)}")
print(f"Columns: {list(all_annot_insect.columns)}")
all_annot_insect.head()

valid_annot_insect = all_annot_insect[(all_annot_insect["species_name"] == "Omocestus_viridulus") & (all_annot_insect["subset"] == "Train")]
print(len(valid_annot_insect))


In [ ]:
# ── Detect overlapping annotations (details no overlapping annotations unlike ECO) ──────
labels_per_file_insect = all_annot_insect.groupby("file_name")["species_name"].count()
print(f"{'Average annotations per file:':<30} {labels_per_file_insect.mean()}")
print(f"{'Least annotations per file:':<30} {labels_per_file_insect.min()}")
print(f"{'Least annotated file:':<30} {labels_per_file_insect.idxmin()}")
print(f"{'Maximum annotations per file:':<30} {labels_per_file_insect.max()}")
print(f"{'Most annotated file:':<30} {labels_per_file_insect.idxmax()}")


In [ ]:
# ── Quantify Orthoptera annotations in the dataset ────────────────────────────────
orth_insect = all_annot_insect[all_annot_insect["group"] == "Orthoptera"]
print(f"Orthoptera species: {orth_insect['species_name'].nunique()}\n")
print(f"Annotations per species:\n\n{orth_insect['species_name'].value_counts().to_string()}")


In [ ]:
# ── Filter to UK Orthoptera species ────────────────────────────────
UK_SPECIES_INSECT = [
    "Chorthippus_brunneus",
    "Pseudochorthippus_parallelus",
    "Omocestus_viridulus",
    "Tettigonia_viridissima",
    "Roeseliana_roeselii",
    "Pholidoptera_griseoaptera",
    "Leptophyes_punctatissima",
    "Gryllus_campestris"
]

orth_uk_insect = orth_insect[orth_insect["species_name"].isin(UK_SPECIES_INSECT)].copy()
print(f"InsectSet459 UK species annotations: {len(orth_uk_insect)}")
print(f"InsectSet459 UK species clips: {orth_uk_insect['file_name'].nunique()}")
print(f"\nAnnotations per species:\n\n{orth_uk_insect['species_name'].value_counts().to_string()}\n")


In [ ]:
# ── Detail sample rates and durations ────────────────────────────────

insect_file_index = {f.name: f for f in INSECT_ROOT.rglob("*") if f.suffix in (".wav", ".mp3")}
orth_uk_insect["filepath"] = orth_uk_insect["file_name"].map(insect_file_index)

# Derive unique sample rates in dataset and count how often each sample rate appears.
srs = {}
for fp in orth_uk_insect["filepath"].unique():
    y, sr = librosa.load(fp, sr=None)
    if sr in srs:
        srs[sr] += 1
    else:
        srs[sr] = 1

print("Sample Rates:\n")
for sr in srs:
    print(f"{sr}hz: {srs[sr]}")

print(f"\nMax clip duration: {orth_uk_insect['filepath'].apply(lambda fp: sf.info(fp).duration).max()}")
print(f"Minimum clip duration: {orth_uk_insect['filepath'].apply(lambda fp: sf.info(fp).duration).min()}")
print(f"Average clip duration: {orth_uk_insect['filepath'].apply(lambda fp: sf.info(fp).duration).mean()}")


In [ ]:
# ── Inspect a clip — spectrogram preview ─────────────────────────────────────
# Set use_random to True for a randomised UK Orthoptera sample
# Set use_random to False to select any annotation from InsectSet459
use_random = False
selected_index = 20720  # CSV row number minus 1

if use_random:
    sample = orth_uk_insect.sample(1).iloc[0]
else:
    sample = all_annot_insect.loc[selected_index]

audio_path_insect = next(INSECT_ROOT.rglob(sample["file_name"]), None)

if audio_path_insect and audio_path_insect.exists():
    y, sample_rate = librosa.load(audio_path_insect, sr=None)

    print(f"{'Species:':<20}{sample['species_name']}")
    print(f"{'File:':<20}{sample['file_name']}")
    print(f"{'Sample Rate:':<20}{sample_rate} Hz")
    print(f"Sample rate: {sample_rate} Hz   Duration: {len(y) / sample_rate:.1f} s")

    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    librosa.display.waveshow(y, sr=sample_rate, ax=axes[0])
    axes[0].set_title(f"{sample['species_name']} — waveform")

    n_fft = min(2048, len(y))
    S = librosa.feature.melspectrogram(y=y, sr=sample_rate, n_mels=128, fmax=sample_rate/2, n_fft=n_fft)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sample_rate, x_axis="time", y_axis="mel",
                             fmax=sample_rate/2, ax=axes[1], cmap="viridis")
    axes[1].set_title("Mel spectrogram")
    plt.colorbar(axes[1].collections[0], ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()
else:
    print(f"Audio file not found: {audio_path}")
    